In [ ]:
import ollama
from loguru import logger

In [2]:
llm_model = "gpt-oss:20b"

In [ ]:
def wikipedia(country: str) -> str:
    """"
    Wikipedia tool function that simulates fetching information about a country from Wikipedia.

    Args:
        country (str): The name of the country to query.
    
    Returns:
        str: A string containing the information retrieved from Wikipedia about the specified country.
    """
    try:
        capitals = {
            'Italy': 'Rome',
            'Spain': 'Madrid',
        }
        capital = capitals.get(country, "Capital not found.")
        print(f"Fetched from Wikipedia: The capital of {country} is {capital}.")
        return capital
    except Exception as e:
        logger.error(f"Error calling Wikipedia API: {e}")
        return "Error fetching data from Wikipedia."

In [ ]:
class Agent:
    
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        self.tools_used = []  # Initialize a list to track tools used
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        try:
            self.messages.append({"role": "user", "content": message})
            result = self.execute()
            self.messages.append({"role": "assistant", "content": result})
            return result
        except Exception as e:
            logger.error(f"Error during agent call: {e}")
            return "An error occurred while processing your request."

    def execute(self):
        try:
            response = ollama.chat(
                    model=llm_model,
                    messages=self.messages,
                    options={"temperature": 0},
                    tools=[wikipedia]
                )
            tool_calls = response["message"].get("tool_calls", [])
            if tool_calls:
                for call in tool_calls:
                    print(f"Tool called: {call.function.name} with arguments {call.function.arguments}")
                    if call.function.name == 'wikipedia':
                        result = wikipedia(call.function.arguments['country'])
                    else:
                        result = 'Unknown tool'
                    self.messages.append({'role': 'tool', 
                                    'tool_name': call.function.name, 
                                    'content': str(result)})
            final_response = ollama.chat(
                    model=llm_model,
                    messages=self.messages,
                    options={"temperature": 0},
                    tools=[wikipedia]
                )
            return final_response["message"]["content"]
        except Exception as e:
            logger.error(f"Error during agent execution: {e}")
            return "An error occurred while processing your request."

In [ ]:
agent = Agent(system="You are a helpful assistant. You can use tools like Wikipedia to answer questions.")

In [7]:
result = agent(message="What is the capital of Italy?")
result

'The capital of Italy is **Rome**.'

In [9]:
response = agent(message="The capital of Spain in wikipedia?")
response

Tool called: wikipedia with arguments {'country': 'Spain'}
Fetched from Wikipedia: The capital of Spain is Madrid.


'According to Wikipedia, the capital of Spain is **Madrid**.'

In [10]:
response = agent(message="Which tools did you use to answer the previous question?")
response

'I used the **Wikipedia tool** to look up the capital of Spain.'

In [11]:
response = agent(message="Thank you!")
response

'You’re welcome! If you have any more questions, feel free to ask.'